# nnU-Net v2 on Google Colab
Select **Runtime > Change runtime type > T4 GPU** before you start.

## 1. Mount Drive and install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q nnunetv2
!nvidia-smi | head -n 15

## 2. Environment variables
Run this cell again after every reconnect.

In [ ]:
import os
base = '/content/drive/MyDrive/nnunet'
for d in ['raw', 'preprocessed', 'results']:
    os.makedirs(f'{base}/{d}', exist_ok=True)

os.environ['nnUNet_raw'] = f'{base}/raw'
os.environ['nnUNet_preprocessed'] = f'{base}/preprocessed'
os.environ['nnUNet_results'] = f'{base}/results'
os.environ['nnUNet_compile'] = 'false'  # optional: no compile warnings

## 3. Save a checkpoint every 10 epochs
Run once per session, before training.

In [ ]:
import nnunetv2, pathlib
f = pathlib.Path(nnunetv2.__file__).parent / 'training/nnUNetTrainer/nnUNetTrainer.py'
f.write_text(f.read_text().replace('self.save_every = 50', 'self.save_every = 10'))
!grep -n "save_every" {f}

## 4. Check the dataset
The folder `Dataset001_IRCADb` must be in `nnunet/raw/`. Create it with `scripts/convert_ircadb_to_nnunet.py`.

In [ ]:
!ls {base}/raw/Dataset001_IRCADb
!cat {base}/raw/Dataset001_IRCADb/dataset.json

## 5. Plan and preprocess
If the worker crashes (out of RAM), use `-np 1`.

In [ ]:
!nnUNetv2_plan_and_preprocess -d 1 --verify_dataset_integrity -c 3d_fullres -np 1 --no_pbar

## 6. Train (and resume)
`--c` resumes from `checkpoint_latest.pth`. Without it, training starts at epoch 0.

In [ ]:
!nnUNetv2_train 1 3d_fullres 0 -device cuda -tr nnUNetTrainer_100epochs --c

## 7. Predict
Put the CT files in `imagesTs`. Name them `case_0000.nii.gz`.

In [ ]:
os.makedirs(f'{base}/imagesTs', exist_ok=True)
!nnUNetv2_predict -i {base}/imagesTs -o {base}/predictions -d 1 -c 3d_fullres -f 0 -tr nnUNetTrainer_100epochs -chk checkpoint_best.pth

## 8. Validation results of fold 0
nnU-Net writes these at the end of training.

In [ ]:
!ls {base}/results/Dataset001_IRCADb/nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres/fold_0/validation
!cat {base}/results/Dataset001_IRCADb/nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres/fold_0/validation/summary.json | head -n 40